# 01 — Training Colab Runner

Run `notebooks/00_preprocess_and_align.ipynb` first. That notebook owns raw-image preprocessing, alignment, filtering, `manifest.csv`, and the Hugging Face dataset export.

This notebook starts after those artifacts already exist and focuses on optional model training.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

## 2. Clone Or Update Repo

In [ ]:
REPO_URL = 'https://github.com/gabeweng/image-style-transfer.git'
REPO_DIR = '/content/image-style-transfer'

import os

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull
else:
    %cd /content
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

## 3. Install Runtime Dependencies

In [ ]:
!pip install -q uv
!uv pip install --system pillow pandas torch torchvision diffusers==0.27.2 'transformers>=4.38.0,<5' 'huggingface-hub<0.26' accelerate peft datasets safetensors xformers wandb tqdm

## 4. Configure Paths

In [ ]:
BASE = '/content/drive/My Drive/CIS_5190_group_project'

MANIFEST_CSV = f'{BASE}/manifest.csv'
FILTERED_DIR = f'{BASE}/filtered_aligned'
HF_DATASET_DIR = f'{BASE}/hf_dataset'
HF_CONTROLNET_DIR = f'{BASE}/data/hf_dataset_controlnet'
CHECKPOINT_DIR = f'{BASE}/checkpoints'

print('Manifest:', MANIFEST_CSV)
print('Filtered images:', FILTERED_DIR)
print('HF dataset:', HF_DATASET_DIR)
print('Optional ControlNet dataset:', HF_CONTROLNET_DIR)
print('Checkpoints:', CHECKPOINT_DIR)

## 5. Validate Preprocess And Align Outputs

In [ ]:
import json
import os
import pandas as pd

assert os.path.exists(MANIFEST_CSV), f'Missing manifest: {MANIFEST_CSV}'
assert os.path.isdir(FILTERED_DIR), f'Missing filtered image directory: {FILTERED_DIR}'
assert os.path.exists(f'{HF_DATASET_DIR}/metadata.jsonl'), f'Missing HF metadata: {HF_DATASET_DIR}/metadata.jsonl'

manifest_df = pd.read_csv(MANIFEST_CSV)
kept_df = manifest_df[manifest_df['status'] == 'kept'].copy() if 'status' in manifest_df.columns else manifest_df.copy()

print(f'Manifest rows: {len(manifest_df)}')
print(f'Kept final images: {len(kept_df)}')
if 'status' in manifest_df.columns:
    print('Status counts:', manifest_df['status'].value_counts().to_dict())
if len(kept_df):
    print('Train/val split:', kept_df['split'].value_counts().to_dict())
    display(kept_df.head())

missing = [p for p in kept_df['file_name'].head(20) if not os.path.exists(os.path.join(BASE, p))]
assert not missing, f'Some kept manifest files were not found: {missing[:5]}'

with open(f'{HF_DATASET_DIR}/metadata.jsonl') as f:
    metadata_rows = sum(1 for _ in f)
print(f'HF metadata rows: {metadata_rows}')
print('Preprocess/alignment artifacts are ready for training.')

## 6. GPU Check

Run the remaining training and inference sections on a GPU runtime.

In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 7. Optional W&B Tracking

Set `USE_WANDB = True` if you want diffusion training metrics and checkpoints logged to Weights & Biases. You will be prompted to paste your W&B API key.

In [ ]:
USE_WANDB = False
WANDB_PROJECT = 'image-style-transfer'

if USE_WANDB:
    import os
    os.environ['WANDB_PROJECT'] = WANDB_PROJECT
    os.environ['WANDB_LOG_MODEL'] = 'checkpoint'
    !wandb login
    REPORT_TO_ARG = '--report_to=wandb'
else:
    REPORT_TO_ARG = ''

print('W&B enabled:', USE_WANDB)

## 8. Install Diffusers Training Examples

The LoRA and ControlNet training scripts live in the Hugging Face `diffusers` examples directory.

In [ ]:
DIFFUSERS_DIR = '/content/diffusers'

if os.path.exists(DIFFUSERS_DIR):
    %cd {DIFFUSERS_DIR}
    !git fetch --tags
    !git checkout v0.27.2
else:
    %cd /content
    !git clone --branch v0.27.2 --depth 1 https://github.com/huggingface/diffusers.git {DIFFUSERS_DIR}

%cd {REPO_DIR}

## 9. Configure Accelerate

This writes a basic single-GPU config so `accelerate launch` can run without the interactive setup prompt.

In [ ]:
!accelerate config default

## 10. Train Stable Diffusion LoRA

This writes LoRA weights to `checkpoints/lora`. It saves frequent checkpoints and resumes from the latest checkpoint if Colab disconnects. The Hugging Face training script includes tqdm progress bars by default.

In [ ]:
LORA_DIR = f'{BASE}/checkpoints/lora'

!accelerate launch /content/diffusers/examples/text_to_image/train_text_to_image_lora.py \
  --pretrained_model_name_or_path="stable-diffusion-v1-5/stable-diffusion-v1-5" \
  --train_data_dir="$HF_DATASET_DIR" \
  --output_dir="$LORA_DIR" \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --num_train_epochs=10 \
  --learning_rate=1e-4 \
  --lr_scheduler="cosine" \
  --mixed_precision="fp16" \
  --gradient_checkpointing \
  --checkpointing_steps=100 \
  --checkpoints_total_limit=3 \
  --resume_from_checkpoint="latest" \
  --caption_column="text" \
  $REPORT_TO_ARG

## 11. Train ControlNet

This writes a fine-tuned ControlNet checkpoint to `checkpoints/controlnet`. It saves frequent checkpoints and resumes from the latest checkpoint if Colab disconnects. This is usually more expensive than LoRA training.

In [ ]:
CONTROLNET_DIR = f'{BASE}/checkpoints/controlnet'

!accelerate launch /content/diffusers/examples/controlnet/train_controlnet.py \
  --pretrained_model_name_or_path="stable-diffusion-v1-5/stable-diffusion-v1-5" \
  --output_dir="$CONTROLNET_DIR" \
  --train_data_dir="$HF_CONTROLNET_DIR" \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=2 \
  --num_train_epochs=5 \
  --mixed_precision="fp16" \
  --gradient_checkpointing \
  --checkpointing_steps=100 \
  --checkpoints_total_limit=3 \
  --resume_from_checkpoint="latest" \
  --conditioning_image_column="conditioning_images" \
  --image_column="images" \
  --caption_column="text" \
  $REPORT_TO_ARG

## 12. Final Training Artifacts

In [ ]:
expected_outputs = [
    f'{HF_DATASET_DIR}/metadata.jsonl',
    f'{CHECKPOINT_DIR}/lora',
]

if os.path.exists(HF_CONTROLNET_DIR):
    expected_outputs.append(f'{CHECKPOINT_DIR}/controlnet')

for path in expected_outputs:
    print(('OK     ' if os.path.exists(path) else 'MISSING'), path)